<a href="https://colab.research.google.com/github/wallafsilva03-spec/Scintila-o-comite-/blob/claude%2Ffrota-equipamento-rtk-metrics/baixar_scintillation_(5).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INPE Scintillation S4 - Consolidador por Hora
Baixa os arquivos, ignora os `-1` (sem dado), tira a **média** dos valores de cintilação,
agrupa por **hora** e classifica pela **escala de cintilação**.

**Saída:** TXT com colunas `Data | Hora | Cintilacao_Media | Escala`

In [14]:
!pip install requests beautifulsoup4 -q

## CONFIGURAÇÃO
Escolha o intervalo de dias. **Comece pequeno** (ex: 1 a 3) para testar!
Depois, se quiser tudo, use 1 a 175 (mas vai demorar bastante).

In [15]:
DIA_INICIO = 130   # primeira pasta (1 = 001)
DIA_FIM    = 175  # última pasta (mude para 175 quando quiser tudo)

BASE = 'https://embracedata.inpe.br/scintillation/maps/s4/2026/'

## Funções auxiliares

In [16]:
import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict
from statistics import mean

def escala_cintilacao(s4):
    """Classifica o valor de S4 na escala de cintilação"""
    if s4 < 0.2:
        return 'Sem cintilacao'
    elif s4 < 0.4:
        return 'Fraca'
    elif s4 < 0.6:
        return 'Moderada'
    else:
        return 'Forte'

def listar_arquivos(pasta):
    """Lista os nomes dos arquivos TXT dentro de uma pasta de dia"""
    url = BASE + pasta + '/'
    r = requests.get(url, timeout=20)
    soup = BeautifulSoup(r.text, 'html.parser')
    arquivos = []
    for a in soup.find_all('a', href=True):
        href = a['href']
        if href.endswith('.txt'):
            arquivos.append(url + href)
    return arquivos

def maximo_do_arquivo(url):
    """Baixa um arquivo e retorna (data, hora, maximo_s4). Ignora valores -1."""
    try:
        r = requests.get(url, timeout=20)
        nome = url.split('/')[-1]  # S4_MAP_20260101_0000.txt
        partes = nome.replace('.txt', '').split('_')
        data = partes[2]      # 20260101
        hora = partes[3]      # 0000 (HHMM)

        # Lê todos os números, ignora -1 (sem dado)
        valores = []
        for linha in r.text.split('\n'):
            for v in linha.split(';'):
                v = v.strip()
                if v and v != '-1':
                    try:
                        num = float(v)
                        if num >= 0:
                            valores.append(num)
                    except ValueError:
                        pass

        if valores:
            maximo = max(valores)   # PICO de cintilação no mapa
        else:
            maximo = None  # arquivo sem nenhum dado válido
        return (data, hora, maximo)
    except Exception:
        return None

## Baixar e processar (pode demorar)

In [17]:
# 1) Coleta a lista de todos os arquivos do intervalo escolhido
todas_urls = []
for dia in range(DIA_INICIO, DIA_FIM + 1):
    pasta = f'{dia:03d}'
    arquivos = listar_arquivos(pasta)
    todas_urls.extend(arquivos)
    print(f'Pasta {pasta}: {len(arquivos)} arquivos')

print(f'\nTotal de arquivos a baixar: {len(todas_urls)}')

Pasta 130: 1440 arquivos
Pasta 131: 1440 arquivos
Pasta 132: 1440 arquivos
Pasta 133: 1440 arquivos
Pasta 134: 1440 arquivos
Pasta 135: 1440 arquivos
Pasta 136: 1440 arquivos
Pasta 137: 1440 arquivos
Pasta 138: 1440 arquivos
Pasta 139: 1440 arquivos
Pasta 140: 1440 arquivos
Pasta 141: 1440 arquivos
Pasta 142: 1440 arquivos
Pasta 143: 1440 arquivos
Pasta 144: 1440 arquivos
Pasta 145: 1440 arquivos
Pasta 146: 1440 arquivos
Pasta 147: 1440 arquivos
Pasta 148: 1440 arquivos
Pasta 149: 1440 arquivos
Pasta 150: 1440 arquivos
Pasta 151: 1440 arquivos
Pasta 152: 1440 arquivos
Pasta 153: 1440 arquivos
Pasta 154: 1440 arquivos
Pasta 155: 1440 arquivos
Pasta 156: 1440 arquivos
Pasta 157: 1440 arquivos
Pasta 158: 1440 arquivos
Pasta 159: 1440 arquivos
Pasta 160: 1440 arquivos
Pasta 161: 1440 arquivos
Pasta 162: 1440 arquivos
Pasta 163: 1440 arquivos
Pasta 164: 1440 arquivos
Pasta 165: 1440 arquivos
Pasta 166: 1440 arquivos
Pasta 167: 1440 arquivos
Pasta 168: 1440 arquivos
Pasta 169: 1440 arquivos


In [18]:
from concurrent.futures import ThreadPoolExecutor

# 2) Baixa em paralelo (10 ao mesmo tempo) para ir mais rápido
resultados = []
concluidos = 0

with ThreadPoolExecutor(max_workers=10) as executor:
    for res in executor.map(maximo_do_arquivo, todas_urls):
        if res is not None:
            resultados.append(res)
        concluidos += 1
        if concluidos % 200 == 0:
            print(f'{concluidos}/{len(todas_urls)} processados...')

print(f'\nConcluído! {len(resultados)} arquivos com resultado.')

200/66240 processados...
400/66240 processados...
600/66240 processados...
800/66240 processados...
1000/66240 processados...
1200/66240 processados...
1400/66240 processados...
1600/66240 processados...
1800/66240 processados...
2000/66240 processados...
2200/66240 processados...
2400/66240 processados...
2600/66240 processados...
2800/66240 processados...
3000/66240 processados...
3200/66240 processados...
3400/66240 processados...
3600/66240 processados...
3800/66240 processados...
4000/66240 processados...
4200/66240 processados...
4400/66240 processados...
4600/66240 processados...
4800/66240 processados...
5000/66240 processados...
5200/66240 processados...
5400/66240 processados...
5600/66240 processados...
5800/66240 processados...
6000/66240 processados...
6200/66240 processados...
6400/66240 processados...
6600/66240 processados...
6800/66240 processados...
7000/66240 processados...
7200/66240 processados...
7400/66240 processados...
7600/66240 processados...
7800/66240 proce

In [19]:
# 3) Agrupa por DATA + BLOCO DE 30 MIN e calcula MÉDIA e MÁXIMO
grupos = defaultdict(list)

for data, hora, valor in resultados:
    if valor is not None:
        hh = hora[:2]
        mm = int(hora[2:])
        bloco = '00' if mm < 30 else '30'
        grupos[(data, hh, bloco)].append(valor)

linhas_finais = []

for (data, hh, bloco) in sorted(grupos.keys()):
    valores = grupos[(data, hh, bloco)]

    media_bloco = mean(valores)
    max_bloco   = max(valores)

    data_fmt = f'{data[:4]}-{data[4:6]}-{data[6:8]}'

    escala_media = escala_cintilacao(media_bloco)
    escala_max   = escala_cintilacao(max_bloco)

    linhas_finais.append(
        f'{data_fmt};{hh}:{bloco};'
        f'{media_bloco:.4f};'
        f'{max_bloco:.4f};'
        f'{escala_media};'
        f'{escala_max}'
    )

print(f'Total de linhas (blocos de 30 min): {len(linhas_finais)}')
print('\nPrévia:')
print('Data;Hora;Cintilacao_Media;Cintilacao_Maxima;Escala_Media;Escala_Maxima')
for l in linhas_finais[:10]:
    print(l)

Total de linhas (blocos de 30 min): 2208

Prévia:
Data;Hora;Cintilacao_Media;Cintilacao_Maxima;Escala_Media;Escala_Maxima
2026-05-10;00:00;0.1452;0.2090;Sem cintilacao;Fraca
2026-05-10;00:30;0.2056;0.3510;Fraca;Fraca
2026-05-10;01:00;0.1517;0.2560;Sem cintilacao;Fraca
2026-05-10;01:30;0.1555;0.2850;Sem cintilacao;Fraca
2026-05-10;02:00;0.1961;0.2880;Sem cintilacao;Fraca
2026-05-10;02:30;0.1632;0.2160;Sem cintilacao;Fraca
2026-05-10;03:00;0.1567;0.2000;Sem cintilacao;Fraca
2026-05-10;03:30;0.2184;0.5050;Fraca;Moderada
2026-05-10;04:00;0.2009;0.2630;Fraca;Fraca
2026-05-10;04:30;0.1958;0.2600;Sem cintilacao;Fraca


In [20]:
# 4) Salva o TXT consolidado
nome_saida = 'cintilacao_consolidada.txt'
with open(nome_saida, 'w', encoding='utf-8') as f:
    f.write('Data;Hora;Cintilacao_Media;Cintilacao_Maxima;Escala_Media;Escala_Maxima\n')
    f.write('\n'.join(linhas_finais))

print(f'Arquivo salvo: {nome_saida}')

Arquivo salvo: cintilacao_consolidada.txt


In [21]:
# 5) Baixa o arquivo para o seu computador
from google.colab import files
files.download(nome_saida)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>